# 16 — Technical Discussion

## Objective

Provide a **Q&A** resource that connects post-training theory and system design to the executed BioLLM experiments. Mathematical objectives describe established methods; numeric claims below refer only to saved local evidence. Notebook 16 reads prior artifacts and performs no model training or final-test inference.

## Evidence map

The evidence map follows Supervised Fine-Tuning (SFT), Direct Preference Optimization (DPO), Group Relative Policy Optimization (GRPO), Proximal Policy Optimization (PPO), and Machine Learning Operations (MLOps) through their separate notebook results.

- [Dataset and split design](03_dataset_design.ipynb), [baseline evaluation](04_baseline_evaluation.ipynb), [SFT training](05_qlora_sft.ipynb), [SFT evaluation](06_sft_evaluation.ipynb).
- [Preference data](07_preference_data.ipynb), [DPO](08_dpo_training.ipynb), [reward model](09_reward_modeling.ipynb), [synthetic PPO](10_rlhf_experiment.ipynb), [GRPO](11_grpo_training.ipynb), [comparison](12_comparative_evaluation.ipynb).
- [Export](13_export_and_ollama.ipynb), [illustrative integration](14_oncolncai_integration_demo.ipynb), [MLOps evidence](15_mlops_pipeline.ipynb), [research protocol](../RESEARCH_PROTOCOL.md), and [experiment status](../EXPERIMENT_STATUS.md).

The common base is `Qwen/Qwen2.5-0.5B-Instruct` at revision `7ae557604adf67be50417f59c2c2f167def9a775`. The PubMedQA labeled split contains 775 training, 121 validation, and 104 sealed final-test records. The latter has not been scored.


## Q&A — Transformer and Large Language Model fundamentals

### 1. What does a causal Transformer learn, and why does attention matter?

**Answer.** A causal Large Language Model (LLM) estimates the next-token distribution $p_\theta(x_t\mid x_{<t})$ and maximizes $\sum_t\log p_\theta(x_t\mid x_{<t})$. A masked attention head computes $\operatorname{softmax}(QK^\top/\sqrt{d_k}+M)V$, where $M$ prevents access to future tokens. Multiple layers mix contextual information, but next-token likelihood alone does not establish factuality or clinical validity. Chat templates insert role boundaries that affect tokenization and therefore the actual conditional distribution. **Project evidence:** [notebook 00](00_project_overview.ipynb), [notebook 01](01_environment_and_gpu.ipynb), and the common-base configuration in [MLOps](15_mlops_pipeline.ipynb). **Failure mode:** evaluating an adapter with a different template, tokenizer, or base revision confounds the comparison.

### 2. Why can generation configuration change an apparent benchmark result?

**Answer.** Greedy decoding selects the highest-probability next token at each step, while sampling adds stochasticity; maximum token count and stop rules can alter whether an answer parses. Notebook 12 used the same chat template, greedy decoding, `max_new_tokens=8`, and strict `yes/no/maybe` parser for all four variants. Its 100% valid-format rate measures parser compliance, not biomedical correctness. **Project evidence:** [comparison notebook](12_comparative_evaluation.ipynb) and `../runs/mlops_15/experiment_report.json` (local experiment report). **Failure mode:** comparing models under unequal prompts or decoding budgets.


## Q&A — Supervised Fine-Tuning and Quantized Low-Rank Adaptation

### 3. What is the Supervised Fine-Tuning objective, and what was learned here?

**Answer.** Supervised Fine-Tuning (SFT) minimizes completion-token negative log likelihood, $L_{\mathrm{SFT}}=-\sum_{t\in\mathrm{assistant}}\log p_\theta(y_t\mid x,y_{<t})$; prompt tokens should not become target labels when completion-only learning is intended. The 200 selected PubMedQA training IDs produced a 40-step adapter. Its validation accuracy was 73/121 (60.3%) versus 19/121 (15.7%) for the base, but only 9 more correct than the 64/121 majority-yes rule; the paired SFT-versus-majority exact $p\approx0.188$. Maybe-class recall was zero. **Evidence:** [training](05_qlora_sft.ipynb), [evaluation](06_sft_evaluation.ipynb), [matched comparison](12_comparative_evaluation.ipynb). **Failure mode:** treating training loss 0.3447 as proof of reliable three-class generalization.

### 4. How do Low-Rank Adaptation and Quantized Low-Rank Adaptation differ in this run?

**Answer.** Low-Rank Adaptation (LoRA) freezes the base weight $W$ and learns $\Delta W=(\alpha/r)BA$ with low-rank matrices. Quantized Low-Rank Adaptation (QLoRA) additionally stores the frozen base in low-bit form while training adapters. Notebook 05's actual run used an **unquantized bfloat16 base with LoRA** because the local quantization stack was incompatible; its filename describes the intended method, not the executed one. The DPO and GRPO runs also used unquantized LoRA. **Evidence:** [SFT run](05_qlora_sft.ipynb), `../runs/mlops_15/experiment_report.json` (local lineage report), and [PEFT quantization guide](https://huggingface.co/docs/peft/developer_guides/quantization). **Failure mode:** reporting memory or quantization benefits that were not measured.


## Q&A — Direct Preference Optimization

### 5. How does Direct Preference Optimization use chosen and rejected responses?

**Answer.** Direct Preference Optimization (DPO) increases the preferred response's policy log-probability *relative to a fixed reference policy*. With $y_w$ preferred and $y_l$ rejected, the pair loss is

$$
L_{\mathrm{DPO}}=-\log\sigma\!\left(\beta\left[\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}-\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\right]\right).
$$

The project started from SFT, used 120 distinct-prompt, length-screened Med-REFL-DPO pairs and 30 optimizer steps. Logged in-batch preference accuracy 0.575 is a training diagnostic. On PubMedQA validation, DPO scored 77/121 versus SFT's 73/121; only four paired cases changed in its favor ($p=0.125$), so incremental improvement is uncertain. **Evidence:** [pair selection](07_preference_data.ipynb), [DPO](08_dpo_training.ipynb), [comparison](12_comparative_evaluation.ipynb). **Failure modes:** reversing pair labels, truncating one side, using mismatched prompt masks, or interpreting in-batch ranking as independent preference generalization.

### 6. Why does the reference policy matter?

**Answer.** The log-ratio measures a policy change relative to the frozen reference; a changed checkpoint changes the objective. Both policy and reference must score the same formatted prompt and response tokens. The project used SFT-derived lineage for DPO rather than substituting a different base. **Evidence:** [DPO implementation](08_dpo_training.ipynb) and [lineage audit](15_mlops_pipeline.ipynb). **Failure mode:** silently moving the reference or switching base revisions makes runs scientifically incomparable.


## Q&A — Reward modeling and preference data

### 7. What does pairwise reward training prove, and what happened here?

**Answer.** A scalar reward model $r_\phi(x,y)$ often minimizes $-\log\sigma(r_\phi(x,y_w)-r_\phi(x,y_l))$. It learns relative rankings in the labeled distribution; the absolute reward is not inherently calibrated. Notebook 09's original 25-step run moved an exposed 61-pair holdout from 29 to 31 correct (47.5% to 50.8%). A 24-pair training-only diagnostic fit all 24, confirming the pipeline can learn those labels. Longer runs reached 60.1% and 57.9% training accuracy on 183 pairs but both scored only 9/22 (40.9%) on a separate group-disjoint preference test, versus 11/22 for the starting adapter. **Evidence:** [reward audit](09_reward_modeling.ipynb) and `../runs/mlops_15/experiment_report.json` (local MLOps report). **Interpretation:** memorization or partial training fit did not generalize; this reward model is not a validated Reinforcement Learning from Human Feedback (RLHF) signal. **Failure mode:** selecting or celebrating a checkpoint by train margin alone.

### 8. Why are preference labels and split design particularly difficult here?

**Answer.** Med-REFL-DPO contains automated response rankings; 21,326 available rows collapse to 355 exact prompt groups, with a largest group of 5,400. Pair-level random splitting could leak a prompt across partitions. Notebook 09 grouped by prompt, screened truncation, and reserved a new test; ambiguous quality and response-length/style cues remain. **Evidence:** [preference inspection](07_preference_data.ipynb), [reward audit](09_reward_modeling.ipynb), [research protocol](../RESEARCH_PROTOCOL.md). **Failure mode:** silently relabeling difficult pairs using the model's own scores or tuning on the exposed holdout.


## Q&A — Reinforcement Learning from Human Feedback and Proximal Policy Optimization

### 9. What are the stages of RLHF, and which were demonstrated?

**Answer.** A typical RLHF pipeline trains an SFT policy, fits a preference reward model, then optimizes a policy under reward and reference-policy constraints. This project executed SFT and a reward-model experiment, but notebook 09 did not establish a generalizable reward signal. Notebook 10 is a synthetic CPU three-action bandit illustrating a clipped policy update; its expected toy reward rose from 0.4 to about 0.996. It is **not** biomedical policy optimization and did not consume the notebook 09 reward checkpoint. **Evidence:** [reward modeling](09_reward_modeling.ipynb), [toy RLHF experiment](10_rlhf_experiment.ipynb), [experiment status](../EXPERIMENT_STATUS.md). **Failure mode:** describing a successful synthetic bandit as successful medical RLHF alignment.

### 10. What does Proximal Policy Optimization clipping do?

**Answer.** Proximal Policy Optimization (PPO) uses an importance ratio $\rho_t=\pi_\theta(a_t\mid s_t)/\pi_{\mathrm{old}}(a_t\mid s_t)$ and maximizes $\min(\rho_t A_t,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)A_t)$. Clipping limits incentive to move the sampled action probability too far on one update; it is not a hard bound on the entire new policy. A reference-policy Kullback–Leibler (KL) penalty can further discourage drift. **Evidence:** [notebook 10's synthetic update](10_rlhf_experiment.ipynb). **Failure modes:** stale log probabilities, mis-signed advantages, noisy rewards, and assuming a clipped loss guarantees safety.


## Q&A — Group Relative Policy Optimization and reward engineering

### 11. How does Group Relative Policy Optimization estimate advantage?

**Answer.** Group Relative Policy Optimization (GRPO) samples several completions for a prompt and compares each reward with its group's mean. A common normalization is $\hat A_i=(r_i-\bar r)/(s_r+\epsilon)$; exact details of clipping and KL control vary by implementation. This run used groups of four, 40 rollouts, 80 distinct PubMedQA train IDs, and 320 sampled completions. Forty-two groups had informative reward variation; 38 were ties. Validation was 76/121 versus SFT's 73/121, with 12 gains and 9 losses ($p\approx0.664$), and lower macro-F1. **Evidence:** [GRPO implementation](11_grpo_training.ipynb), [comparison](12_comparative_evaluation.ipynb). **Failure mode:** treating training reward 0.628 or sample accuracy 185/320 as independent validation.

### 12. How can reward engineering invite reward hacking?

**Answer.** A policy can optimize the measurable proxy rather than the intended property. An exact label/format reward may produce valid `yes/no/maybe` strings while missing medically important distinctions. Here SFT, DPO, and GRPO all achieved valid output format on validation but zero recall for `maybe`; GRPO predicted `yes` for 103 of 121 cases. A reward for citation-like text could similarly produce unsupported citations. **Evidence:** [GRPO reward](11_grpo_training.ipynb), [prediction analysis](12_comparative_evaluation.ipynb), [illustrative evidence controls](14_oncolncai_integration_demo.ipynb). **Mitigation:** inspect classwise errors and adversarial examples, verify cited source spans, and use independent human review for sensitive judgments.


## Q&A — Evaluation, design, and data leakage

### 13. How should the four answer models be compared?

**Answer.** Use matched PubMedQA questions and identical decoding/scoring. The 121-record validation correct counts are base 19, SFT 73, DPO 77, GRPO 76; macro-F1 values are approximately 0.115, 0.422, 0.447, 0.408. SFT's gain over base is large, but the majority-yes baseline already gets 64 correct. DPO's four-case gain over SFT and GRPO's three-case net gain do not establish robust incremental performance. All adapted models have zero `maybe` recall. **Evidence:** [notebook 12](12_comparative_evaluation.ipynb) and `../runs/mlops_15/experiment_report.json` (local saved comparison). **Failure mode:** comparing only accuracy while ignoring class imbalance and paired uncertainty.

### 14. How are validation, preference test, and sealed final test different?

**Answer.** PubMedQA validation (121 records) has been inspected repeatedly for exploratory model comparison. Its 104-record final test remains sealed; it should be evaluated once after a model and analysis plan are fixed. Notebook 09's independent 22-pair test evaluates *reward preference discrimination* on Med-REFL-DPO, not PubMedQA answer accuracy. Prompt grouping limits exact-prompt leakage, but semantic duplicates and unknown upstream revisions remain. **Evidence:** [split notebook](03_dataset_design.ipynb), [reward audit](09_reward_modeling.ipynb), [research protocol](../RESEARCH_PROTOCOL.md). **Failure mode:** tuning against test outcomes or merging metrics across different tasks.


## Q&A — Retrieval-Augmented Generation

### 15. How can Retrieval-Augmented Generation reduce unsupported answers?

**Answer.** Retrieval-Augmented Generation (RAG) retrieves source passages, provides them as constrained context, and asks the generator to cite or abstain when support is absent. Retrieval does not make the source true or guarantee that generated claims follow from it. A useful pipeline records source IDs, retrieval scores, passage text, answer spans, and citation checks. Notebook 04's originating-document proxy has Recall@3 about 0.992, Precision@3 about 0.331, and mean reciprocal rank about 0.988; those are **not** independent human relevance judgments or live vector-index results. Notebook 14 uses invented token-overlap passages and abstains on an unrelated query. **Evidence:** [retrieval baseline](04_baseline_evaluation.ipynb), [illustrative interface](14_oncolncai_integration_demo.ipynb). **Failure modes:** irrelevant retrieved context, stale documents, unsupported citation formatting, and answer leakage into the retrieval corpus.

### 16. What should happen when retrieval is irrelevant or conflicting?

**Answer.** Apply a relevance threshold calibrated on labeled queries, check whether passages actually support the requested claim, surface conflicts with provenance, and abstain when evidence is insufficient. For example, `if score < threshold: return {"answer": None, "reason": "insufficient evidence"}` is a policy sketch, not a calibrated project threshold. Retrieval Recall@$k$ is the share of relevant items recovered in the top $k$; it cannot be inferred from document-origin matching alone. **Evidence:** [notebook 04](04_baseline_evaluation.ipynb) and [notebook 14](14_oncolncai_integration_demo.ipynb). **Failure mode:** forcing an answer solely because vector search returned a nearest neighbor.


## Q&A — LangChain, LangGraph, Model Context Protocol, and agents

### 17. When is LangChain or LangGraph useful?

**Answer.** LangChain supplies composable model, retrieval, and tool interfaces. LangGraph represents a stateful workflow as nodes and conditional edges, useful for branching, interruption, and controlled retries. A checkpointer saves thread state; a store holds cross-thread application data. An in-memory checkpointer does not survive process restart. Notebook 14's own deterministic dispatcher shows boundaries and a step cap, but it does **not** run LangChain or LangGraph. **Evidence:** [notebook 14](14_oncolncai_integration_demo.ipynb), [LangGraph persistence documentation](https://docs.langchain.com/oss/python/langgraph/persistence). **Failure modes:** unbounded loops, stale state, hidden side effects on replay, and assuming framework adoption creates reliable reasoning.

### 18. What makes a tool-using system an agent, and how should it select tools?

**Answer.** An agent selects actions from allowed tools using task state and observations, then decides whether to continue or stop. Tool descriptions are suggestions to a model, not authorization; a separate dispatcher must validate tool name, arguments, scope, and call budget. Notebook 14's fixed dispatcher handles arithmetic and evidence reads, rejects an unallowlisted shell tool, and stops repeated reads. That is an illustrative control-flow check, not an autonomous deployed agent. **Evidence:** [integration demonstration](14_oncolncai_integration_demo.ipynb). **Failure modes:** prompt injection in tool output, recursive calls, destructive actions without approval, and passing unvalidated model text to a shell or database.

### 19. What does Model Context Protocol standardize?

**Answer.** Model Context Protocol (MCP) defines a host/client/server exchange for tools, resources, and prompts. The host decides how a model uses supplied context; the protocol does not itself establish source truth, user consent, or clinical safety. A production host should allowlist servers and methods, scope credentials, validate returned schemas, log provenance, and obtain authorization for consequential operations. No MCP server was executed in notebook 14. **Evidence:** [notebook 14](14_oncolncai_integration_demo.ipynb), [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture), and [HTTP transport authorization specification](https://modelcontextprotocol.io/specification/2025-06-18/basic/authorization). **Failure mode:** treating server availability as permission to invoke every advertised tool.

### 20. When do multiple agents help?

**Answer.** Separate agents can independently retrieve, analyze, and audit evidence when tasks have distinct inputs and checkable outputs. They add model calls, latency, coordination errors, and extra attack surfaces. A single bounded workflow is preferable for notebook 14's simple arithmetic and evidence examples. A possible future split would require explicit roles, shared provenance, conflict resolution, and a final human-readable trace. **Evidence:** [notebook 14](14_oncolncai_integration_demo.ipynb). **Failure mode:** multiplying agreement among agents trained on the same flawed source into false confidence.


## Q&A — Production architecture, inference, security, cost, and monitoring

### 21. What was actually exported, and what remains before serving?

**Answer.** Notebook 13 merged the SFT LoRA adapter into a float32 Hugging Face checkpoint and verified source hashes, a synthetic forward-pass comparison (maximum last-token logit difference $5.17\times10^{-5}$), and exact reload of that probe. It did not create GGUF or Ollama artifacts. Notebook 15 found a valid local checkpoint and local synthetic inference, but `deployment/model/` contains only `.gitkeep`; no service smoke test or Ollama model was verified. Registry `production_model` is `null`. **Evidence:** [export](13_export_and_ollama.ipynb), `../runs/mlops_15/deployment_check.json` (local deployment check), `../runs/mlops_15/registry.json` (local MLOps registry). **Failure mode:** equating serialization or one prompt's logits with end-to-end deployment readiness.

### 22. How should latency and cost be estimated before choosing an architecture?

**Answer.** Measure end-to-end request latency, including retrieval, tool calls, queueing, prompt processing, and generated tokens. Approximate variable model cost as $C\approx N_{\mathrm{in}}c_{\mathrm{in}}+N_{\mathrm{out}}c_{\mathrm{out}}+C_{\mathrm{tools}}$ under the provider's current unit definitions; local GPU cost instead includes utilization, memory, and operations. Cache safe deterministic outputs and cap tokens and tool loops, then profile quality/latency changes. No live cost or throughput benchmark was executed here. **Evidence:** [illustrative workflow](14_oncolncai_integration_demo.ipynb) and [deployment audit](15_mlops_pipeline.ipynb). **Failure mode:** counting only model generation time while tool and retrieval tails dominate.

### 23. What security and monitoring controls matter most?

**Answer.** Treat retrieved text and tool responses as untrusted data. Enforce tool allowlists, typed arguments, least-privilege credentials, data minimization, audit logs, and explicit review for consequential actions. Monitor input mix, parse failures, per-class errors, abstentions, source support, latency, resource use, and rollback events. Notebook 15 specifies regression/monitoring design and an in-memory `DEMO-*` rollback only; it has no live traffic or service telemetry. **Evidence:** [notebook 14](14_oncolncai_integration_demo.ipynb), [MLOps notebook](15_mlops_pipeline.ipynb), `../runs/mlops_15/regression_and_monitoring.json` (local monitoring artifact). **Failure mode:** relying on a good offline accuracy number while production inputs drift or unauthorized tools remain reachable.


## Q&A — Research reproducibility and limitations

### 24. What makes a post-training result reproducible and promotable?

**Answer.** Record exact base revision, source dataset identifiers and content fingerprints, split IDs, formatting, seeds, optimizer settings, adapter lineage, evaluation protocol, and output hashes. The MLOps audit verified cached source checksums and local adapter/export lineage. Its registry separates artifact integrity and completed training from validation evidence, reward-signal validity, final-test completion, and service readiness. All saved variants remain `research_verified`; none is deployment ready. **Evidence:** [notebook 15](15_mlops_pipeline.ipynb), `../runs/mlops_15/experiment_report.json` (local experiment report), `../runs/mlops_15/registry.json` (local registry). **Failure mode:** promoting a checkpoint because it reloads or because training loss fell.

### 25. What is the strongest supported conclusion and the next defensible step?

**Answer.** SFT substantially improved the fixed validation score over the base but did not show a convincing gain over the majority rule and failed the `maybe` class. DPO and GRPO completed training but their small incremental validation differences from SFT are uncertain. The reward-model experiment did not establish held-out preference discrimination and cannot support successful RLHF alignment claims. Freeze an analysis plan independently of the repeatedly viewed validation set, address classwise errors and data quality, and only then use the 104-record PubMedQA final test once. Any clinical or production claim requires additional external validation and an actual service audit. **Evidence:** [experiment status](../EXPERIMENT_STATUS.md), [notebook 12](12_comparative_evaluation.ipynb), [notebook 15](15_mlops_pipeline.ipynb). **Failure mode:** promoting an exploratory result to a confirmatory conclusion.


## Results — Read-only evidence check

The following cells parse saved notebook 15 artifacts. They do not retrain, alter predictions, score the sealed final test, or call a deployment service. The comparison and gate outputs should agree with the cited notebooks and registry.


In [1]:
from pathlib import Path
import json

cwd = Path.cwd().resolve()
root = cwd if (cwd / "exploration" / "notebooks").is_dir() else cwd.parent.parent
report_path = root / "exploration" / "runs" / "mlops_15" / "experiment_report.json"
registry_path = root / "exploration" / "runs" / "mlops_15" / "registry.json"
deployment_path = root / "exploration" / "runs" / "mlops_15" / "deployment_check.json"
assert all(path.is_file() for path in (report_path, registry_path, deployment_path))
report = json.loads(report_path.read_text(encoding="utf-8"))
registry = json.loads(registry_path.read_text(encoding="utf-8"))
deployment = json.loads(deployment_path.read_text(encoding="utf-8"))
assert report["configuration"]["model_id"] == "Qwen/Qwen2.5-0.5B-Instruct"
assert report["source_checks"] == {"preference_cache": "verified", "pubmedqa_cache": "verified"}
print("Base:", report["configuration"]["model_id"])
print("Revision:", report["configuration"]["base_revision"])
print("Cached source checks:", report["source_checks"])
print("Sealed final-test policy:", report["configuration"]["final_test_policy"])


Base: Qwen/Qwen2.5-0.5B-Instruct
Revision: 7ae557604adf67be50417f59c2c2f167def9a775
Cached source checks: {'preference_cache': 'verified', 'pubmedqa_cache': 'verified'}
Sealed final-test policy: sealed; no prediction or selection in notebook 15


In [2]:
# Compare like-for-like PubMedQA validation outputs from the saved report.
for row in report["comparison"]:
    print(f'{row["variant"]:5s} {row["correct"]:2d}/{row["n"]} '
          f'accuracy={row["accuracy"]:.3f} macro-F1={row["macro_f1"]:.3f} '
          f'maybe-recall={row["maybe_recall"]:.3f}')
print("DPO vs SFT:", report["paired_comparisons"]["dpo_vs_sft"])
print("GRPO vs SFT:", report["paired_comparisons"]["grpo_vs_sft"])
for name, row in report["reward_revisions"].items():
    print(f'{name}: train={row["train_accuracy"]:.3f}, '
          f'new preference test={row["new_test_accuracy"]:.3f} '
          f'on {row["new_test_pairs"]} pairs')
assert [r["correct"] for r in report["comparison"]] == [19, 73, 77, 76]
assert all(r["new_test_accuracy"] == 9 / 22 for r in report["reward_revisions"].values())


base  19/121 accuracy=0.157 macro-F1=0.115 maybe-recall=1.000
sft   73/121 accuracy=0.603 macro-F1=0.422 maybe-recall=0.000
dpo   77/121 accuracy=0.636 macro-F1=0.447 maybe-recall=0.000
grpo  76/121 accuracy=0.628 macro-F1=0.408 maybe-recall=0.000
DPO vs SFT: {'accuracy_delta': 0.03305785123966942, 'exact_two_sided_p': 0.125, 'gained': 4, 'lost': 0}
GRPO vs SFT: {'accuracy_delta': 0.024793388429752067, 'exact_two_sided_p': 0.6636238098144531, 'gained': 12, 'lost': 9}
reward_model_revision: train=0.601, new preference test=0.409 on 22 pairs
reward_model_revision_seed44: train=0.579, new preference test=0.409 on 22 pairs


In [3]:
# A saved model is not automatically a production model.
assert report["production_model"] is None
assert not deployment["service_smoke"]
for name in ("sft", "dpo", "grpo", "reward_model"):
    state = report["registry"][name]
    print(f'{name:12s} state={state["state"]}, '
          f'trained={state["training_completed"]}, '
          f'deployment_ready={state["deployment_ready"]}')
assert report["registry"]["reward_model"]["reward_signal_validated"] is False
assert all(not entry["deployment_ready"] for entry in report["registry"].values())
print("Production model:", report["production_model"])
print("Service smoke:", deployment["service_smoke"])
print("Research registry available:", registry_path.name)


sft          state=research_verified, trained=True, deployment_ready=False
dpo          state=research_verified, trained=True, deployment_ready=False
grpo         state=research_verified, trained=True, deployment_ready=False
reward_model state=research_verified, trained=True, deployment_ready=False
Production model: None
Service smoke: False
Research registry available: registry.json


## Interpretation

The evidence cells distinguish complete training runs and valid checkpoint files from independently demonstrated performance. The SFT gain over base is substantial on the fixed validation set, yet class imbalance and zero `maybe` recall constrain its interpretation. DPO and GRPO add no robust matched improvement over SFT; notebook 09's selected reward models fail the separate preference test. The local export is technically reloadable, while service and final-test gates remain closed.

## Key Observations

1. Theory equations explain optimization targets; they do not certify this project's model quality.
2. An executed demonstration is labeled as such: notebook 10 is synthetic PPO, notebook 14 is a deterministic interface, and notebook 15's rollback is in memory.
3. The sealed PubMedQA final test, live deployment, independent relevance judgments, and reliable RLHF reward remain open work.

## Limitations

This synthesis uses saved exploratory evidence. It adds no new model training, independent final-test result, or production integration.
